# IR 0 — Acquire native NIST spectra (pre-work)

This notebook downloads spectra and molecular identities from the [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) using NISTChemPy. It saves native `x_raw`, `y_raw`, `xunits`, `yunits`, SMILES, metadata and reproducible SMARTS labels in `outputs/nist_acquisition/ir_raw_checkpoint.parquet`.

**No signal-type filtering, unit conversion, normalization, cropping, padding or common-grid interpolation happens here.** Those are workshop carpentry decisions in IR 01. The checkpoint is resumable and can be reused without another download. The old standardized `IR_SMILES_functional_groups.parquet`, if present, is a legacy artifact and is no longer consumed by IR 01.

Live collection is off by default. To acquire more spectra, set `RUN_LIVE_COLLECTION = True`; use a pilot before deliberately setting `MAX_COMPOUNDS = None`. Existing checkpoint records are reused.

In [1]:
import sys
from importlib.util import find_spec
from io import StringIO
from pathlib import Path

if find_spec('nistchempy') is None:
    raise ModuleNotFoundError(
        'NISTChemPy is missing from this notebook kernel. Select the workshop .venv kernel, then run: uv sync --locked'
    )

import jcamp
import nistchempy as nist
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm.auto import tqdm

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs/nist_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ID_FILE = DATA_DIR / 'compounds.parquet'
RAW_CHECKPOINT = OUTPUT_DIR / 'ir_raw_checkpoint.parquet'
FAILURE_LOG = OUTPUT_DIR / 'retrieval_failures.parquet'

RUN_LIVE_COLLECTION = False   # Set False to inspect/rebuild an existing checkpoint without network access.
MAX_COMPOUNDS = None           # Pilot batch. Set None for a full, resumable crawl after a successful pilot.
RETRY_PREVIOUS_FAILURES = False  # Keep False to advance through IDs without retrying known no-spectrum records.
CHECKPOINT_EVERY = 25
request_config = nist.RequestConfig(delay=3.0, max_attempts=3, kwargs={'timeout': 30.0})

if nist.__version__ != '2.0.0':
    raise RuntimeError(f'Tested with nistchempy 2.0.0; found {nist.__version__}.')
compound_ids = pd.read_parquet(ID_FILE)['data'].dropna().astype(str).drop_duplicates().tolist()
print(f'NistChemPy {nist.__version__}; {len(compound_ids):,} candidate IDs; live collection={RUN_LIVE_COLLECTION}')


NistChemPy 2.0.0; 68,030 candidate IDs; live collection=False


In [2]:
SMARTS = {
    'Vinylic': '[$([CX3]=[CX3])]', 'Arene': 'c', 'Alcohol': '[#6][OX2H]',
    'Ether': '[OD2]([#6])[#6]', 'Amine': '[NX3;H2,H1;!$(NC=O)]', 'Nitrile': '[NX1]#[CX2]',
    'Aldehyde': '[CX3H1](=O)[#6]', 'Ketone': '[#6][CX3](=O)[#6]',
    'CarboxylAcid': '[OX2H][CX3]=[OX1]', 'Ester': '[#6][CX3](=O)[OX2H0][#6]',
    'Amide': '[NX3][CX3](=[OX1])[#6]', 'AcidChloride': '[CX3](=[OX1])[F,Cl,Br,I]',
}
PATTERNS = {name: Chem.MolFromSmarts(pattern) for name, pattern in SMARTS.items()}

def labels_from_smiles(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError('NIST 2-D molfile produced invalid SMILES.')
    return {name: int(molecule.HasSubstructMatch(pattern)) for name, pattern in PATTERNS.items()}

def fetch_record(compound_id):
    compound = nist.get_compound(compound_id, request_config=request_config)
    if compound is None:
        raise LookupError('No unique NIST record.')
    spectra = compound.get_ir_spectra() or []
    if not spectra:
        raise LookupError('No IR spectrum.')
    parsed = jcamp.jcamp_read(StringIO(spectra[0].jdx_text))
    x = np.asarray(parsed.get('x', []), dtype=float)
    y = np.asarray(parsed.get('y', []), dtype=float)
    molecule = Chem.MolFromMolBlock(compound.get_mol2D(), sanitize=True, removeHs=True)
    if molecule is None:
        raise ValueError('NIST 2-D molfile cannot be parsed.')
    smiles = Chem.MolToSmiles(molecule, canonical=True)
    record = {
        'NIST_molecule': compound_id, 'CAS_NO': compound.cas_rn,
        'Name': parsed.get('title') or compound.name, 'molform': parsed.get('molform') or compound.formula,
        'SMILES': smiles, 'x_raw': x.tolist(), 'y_raw': y.tolist(),
        'xunits': str(parsed.get('xunits', '')).upper(), 'yunits': str(parsed.get('yunits', '')).upper(),
    }
    record.update(labels_from_smiles(smiles))
    return record


In [3]:
if RUN_LIVE_COLLECTION:
    records = pd.read_parquet(RAW_CHECKPOINT).to_dict('records') if RAW_CHECKPOINT.exists() else []
    previous_failures = pd.read_parquet(FAILURE_LOG).to_dict('records') if FAILURE_LOG.exists() else []
    completed = {record['NIST_molecule'] for record in records}
    if not RETRY_PREVIOUS_FAILURES:
        completed.update(record['NIST_molecule'] for record in previous_failures)
    candidates = [identifier for identifier in compound_ids if identifier not in completed]
    if MAX_COMPOUNDS is not None:
        candidates = candidates[:MAX_COMPOUNDS]
    failures = []
    for count, identifier in enumerate(tqdm(candidates, desc='Downloading NIST WebBook IR records'), start=1):
        try:
            records.append(fetch_record(identifier))
        except Exception as error:
            failures.append({'NIST_molecule': identifier, 'error': repr(error)})
        if count % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_parquet(RAW_CHECKPOINT, index=False)
    pd.DataFrame(records).to_parquet(RAW_CHECKPOINT, index=False)
    all_failures = pd.DataFrame([*previous_failures, *failures])
    if all_failures.empty:
        all_failures = pd.DataFrame(columns=['NIST_molecule', 'error'])
    else:
        all_failures = all_failures.drop_duplicates('NIST_molecule', keep='last')
    all_failures.to_parquet(FAILURE_LOG, index=False)
    all_failures.to_csv(OUTPUT_DIR / 'retrieval_failures.csv', index=False)
    print(f'Raw checkpoint: {len(records):,} records; failures this run: {len(failures):,}; retained failures: {len(all_failures):,}.')
else:
    print('Live collection is disabled. Existing raw Parquet, if present, can still be reloaded below.')


Live collection is disabled. Existing raw Parquet, if present, can still be reloaded below.


In [4]:
if RAW_CHECKPOINT.exists():
    native = pd.read_parquet(RAW_CHECKPOINT)
    print(f'Reloaded {len(native):,} native spectra; no data carpentry applied.')
    display(native.groupby(['xunits', 'yunits'], dropna=False).size().rename('spectra').to_frame())
    display(native[['NIST_molecule', 'SMILES', 'xunits', 'yunits', 'x_raw', 'y_raw']].head(3))
else:
    print('No checkpoint yet. Enable RUN_LIVE_COLLECTION for a pilot collection.')

Reloaded 11,001 native spectra; no data carpentry applied.


spectra
xunits      yunits                
1/CM        ABSORBANCE        6497
            REFLECTANCE          1
            TRANSMITTANCE     2613
MICROMETERS TRANSMITTANCE     1890

,NIST_molecule,SMILES,xunits,yunits,x_raw,y_raw
0,C106365,CCCOC(=O)CC,1/CM,ABSORBANCE,"[450.0, 454.0, 458.0, 462.0, 466.0, 470.0, 474...","[0.0, 0.0457976, 0.0435968, 0.0590023999999999..."
1,C91342335,CCC(C)NC1CCCCC1.Cl,MICROMETERS,TRANSMITTANCE,"[2.013, 2.0178898, 2.0227796, 2.0276694, 2.032...","[0.932, 0.932, 0.932, 0.932, 0.932, 0.932, 0.9..."
2,C1569604,CC(C)=CCCC(C)O,1/CM,ABSORBANCE,"[550.0, 554.0, 558.0, 562.0, 566.0, 570.0, 574...","[0.0006222, 0.0004114, 0.0002346, 0.0002363, 0..."


In [5]:
if RAW_CHECKPOINT.exists():
    required = {'NIST_molecule', 'SMILES', 'x_raw', 'y_raw', 'xunits', 'yunits'} | set(SMARTS)
    if not required.issubset(native.columns):
        raise ValueError(f'Native checkpoint missing columns: {sorted(required - set(native.columns))}')
    print('Native schema verified. Open IR 01 to examine units, coverage, invalid arrays and labels.')

Native schema verified. Open IR 01 to examine units, coverage, invalid arrays and labels.
